In [ ]:
# Import packages
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from spatialdata_io import visium
import os
import numpy as np
import anndata as ad
#import igraph
import leidenalg
from scipy import stats
from IPython.display import display

import json
from spatialdata import SpatialData
from spatialdata.models import Image2DModel
from spatialdata_io import visium
import spatialdata_plot
from matplotlib.lines import Line2D

In [ ]:
# Set working directory
working_dir = os.getcwd()
data_path = working_dir
print(data_path)

# Load Visium spatial data object
data_path = os.path.join(working_dir, 'output/Visium_final.zarr')
sdata = SpatialData.read(data_path)

# Visualisations

In [ ]:
sdata.pl.render_images(elements="GSM5764414_JBO001_hires_image").pl.render_shapes(elements="GSM5764414_JBO001", color="part").pl.show("downscaled_hires")
ax = plt.gca()
ax.set_xlim(400,1650)
ax.set_ylim(1550, 750) 
ax.set_title("Spots colored by part",fontsize=16, pad=12)

In [ ]:
sdata.pl.render_images(elements="GSM5764414_JBO001_hires_image").pl.render_shapes(elements="GSM5764414_JBO001", color="dominant_celltype_visium_rctd_2x2").pl.show("downscaled_hires")
ax = plt.gca()
ax.set_xlim(400,1650)
ax.set_ylim(1550, 750) 
ax.set_title("Dominant celltype rctd", fontsize=12, pad=15)

In [ ]:
sdata.pl.render_images(elements="GSM5764414_JBO001_hires_image").pl.render_shapes(elements="GSM5764414_JBO001", color="match_visium_rctd").pl.show("downscaled_hires")
sdata.pl.render_images(elements="GSM5764414_JBO001_hires_image").pl.render_shapes(elements="GSM5764414_JBO001", color="part").pl.show("downscaled_hires")

In [ ]:
print(sdata)
sdata.pl.render_images(elements="GSM5764414_JBO001_hires_image")\
      .pl.render_shapes(elements="GSM5764414_JBO001", color="random_part")\
      .pl.show("downscaled_hires")

ax = plt.gca()
ax.set_xlim(400,1650)
ax.set_ylim(1550, 750) 
ax.set_title("Training in batches", fontsize=16, pad=12)
#plt.show()

In [ ]:
sdata.pl.render_images(elements="GSM5764414_JBO001_hires_image")\
      .pl.render_shapes(elements="GSM5764414_JBO001", color="match_visium_rctd_cat")\
      .pl.show("downscaled_hires")

ax = plt.gca()
ax.set_xlim(400,1055)
ax.set_ylim(1215, 750) 
ax.set_title("RCTD", fontsize=16, pad=12)
ax.vlines(x=1050, ymin=0, ymax=2000, color="black", linestyle="--")
ax.hlines(y=1210, xmin=0, xmax=2000, color="black", linestyle="--")

# Custom legend entries
custom_legend = [    Line2D([0], [0], marker='o', color='w', label='Match',
           markerfacecolor='darkorange', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Mismatch',
           markerfacecolor='dodgerblue', markersize=10)
]

# Add legend to the plot
ax.legend(handles=custom_legend, loc='upper left', frameon=True, fontsize = 15)

#plt.show()

In [ ]:
sdata.pl.render_images(elements="GSM5764414_JBO001_hires_image")\
      .pl.render_shapes(elements="GSM5764414_JBO001", color="match_visium_cell2loc_cat")\
      .pl.show("downscaled_hires")

ax = plt.gca()
ax.set_xlim(400,1055)
ax.set_ylim(1215, 750) 
ax.set_title("Cell2location", fontsize=16, pad=12)
ax.vlines(x=1050, ymin=0, ymax=2000, color="black", linestyle="--")
ax.hlines(y=1210, xmin=0, xmax=2000, color="black", linestyle="--")

# Custom legend entries
custom_legend = [    Line2D([0], [0], marker='o', color='w', label='Match',
           markerfacecolor='darkorange', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Mismatch',
           markerfacecolor='dodgerblue', markersize=10)
]

# Add legend to the plot
ax.legend(handles=custom_legend, loc='upper left', frameon=True, fontsize = 15)
#plt.show()

In [ ]:
sdata.pl.render_images(elements="GSM5764414_JBO001_hires_image")\
      .pl.render_shapes(elements="GSM5764414_JBO001", color="match_visium_cell2loc_2x2_run2_cat")\
      .pl.show("downscaled_hires")

ax = plt.gca()
ax.set_xlim(400,1055)
ax.set_ylim(1215, 750) 
ax.set_title("Run-to-run consistency of cell2location", fontsize=16, pad=12)
ax.vlines(x=1050, ymin=0, ymax=2000, color="black", linestyle="--")
ax.hlines(y=1210, xmin=0, xmax=2000, color="black", linestyle="--")

# Custom legend entries
custom_legend = [    Line2D([0], [0], marker='o', color='w', label='Match',
           markerfacecolor='darkorange', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Mismatch',
           markerfacecolor='dodgerblue', markersize=10)
]

# Add legend to the plot
ax.legend(handles=custom_legend, loc='upper left', frameon=True, fontsize = 15)

#plt.show()

In [ ]:
adata = sdata.table
print(adata.uns)
print(sdata.table)
print(sdata.table.obsm["visium_nnls"].columns)

In [ ]:
spot_counts = sdata.table.obs.groupby("part").size().reset_index(name="Number of spots")
display(spot_counts)

In [ ]:
for method in ["rctd", "nnls", "cell2loc"]:
    for i in sdata.table.obsm["visium_nnls"].columns:
        sdata.table.obsm[f"ratio_cell_types_{method}"] = sdata.table.obsm[f"visium_{method}"].iloc[:,:-1]/sdata.table.obsm[f"visium_{method}_2x2"].iloc[:,:-1]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

for method in ["rctd", "nnls"]:
    key = f"ratio_cell_types_{method}"
    df = sdata.table.obsm[key]

    n_cols = 2
    n_rows = int(np.ceil(len(df.columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows * 3))
    axes = axes.flatten()

    for idx, col in enumerate(df.columns):
        data = df[col].astype(float)

        mean_value = np.mean(data)
        std_dev = np.std(data)
        median = np.median(data)
        t_stat, p_value = stats.ttest_1samp(data, 1)

        bin_range = max(abs(data.min() - 1), abs(data.max() - 1))
        bins = np.linspace(1 - bin_range, 1 + bin_range, 30)

        ax = axes[idx]
        ax.hist(data, bins=bins, edgecolor="black", alpha=0.7)
        ax.set_title(f'{col}', fontsize=12)

        ax.text(
            0.95, 0.85,
            f'Mean: {mean_value:.2f}\nMedian: {median:.2f}\nStd: {std_dev:.2f}\np: {p_value:.2e}',
            transform=ax.transAxes,
            fontsize=10,
            verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(facecolor='white', alpha=0.5, edgecolor='black')
        )

    # Hide unused subplots if any
    for k in range(idx + 1, len(axes)):
        axes[k].axis('off')

    plt.suptitle(f"Histogram per cell type - {method}", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

methods = ["rctd","nnls"]

# Neem de lijst van celtypes aan uit de eerste methode
first_key = f"ratio_cell_types_{methods[0]}"
cell_types = sdata.table.obsm[first_key].columns

n_cols = 2
n_rows = int(np.ceil(len(cell_types) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows * 3))
axes = axes.flatten()

for idx, cell_type in enumerate(cell_types):
    ax = axes[idx]

    for method in methods:
        key = f"ratio_cell_types_{method}"
        df = sdata.table.obsm[key]
        data = df[cell_type].astype(float)

        mean_value = np.mean(data)
        std_dev = np.std(data)
        median = np.median(data)
        t_stat, p_value = stats.ttest_1samp(data, 1)

        bin_range = max(abs(data.min() - 1), abs(data.max() - 1))
        bins = np.linspace(1 - bin_range, 1 + bin_range, 30)

        if method == "rctd":
            ax.hist(data, bins=bins, alpha=0.5, label=method, edgecolor='black')
        else:
            ax.hist(data, bins=bins, alpha=0.5,  color = 'red', label=method, edgecolor='red')

    ax.set_title(f'{cell_type}', fontsize=12)
    ax.legend(fontsize=9)
    

# Verberg ongebruikte subplot-assen
for k in range(len(cell_types), len(axes)):
    axes[k].axis('off')

plt.suptitle("Histogram of ratios per cell type", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()



In [ ]:

methods = ["nnls", "rctd", "cell2loc"]


part0_mask = sdata.table.obs["part"] == 0

first_key = f"visium_{methods[0]}"
cell_types = sdata.table.obsm[first_key].columns[:-1]

n_cols = 2
n_rows = int(np.ceil(len(cell_types) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(8, n_rows * 4))
axes = axes.flatten()


legend_entries = {}

for idx, cell_type in enumerate(cell_types):
    ax = axes[idx]

    all_x = []
    all_y = []

    for method in methods:
        key_visium = f"visium_{method}"
        key_split = f"visium_{method}_2x2"

        df_visium = sdata.table.obsm[key_visium].iloc[part0_mask.values, :-1]
        df_split = sdata.table.obsm[key_split].iloc[part0_mask.values, :-1]

        x = df_visium[cell_type].astype(float)
        y = df_split[cell_type].astype(float)

        all_x.append(x)
        all_y.append(y)

        sc = ax.scatter(x, y, alpha=0.6, label=method, s=10)

        if method not in legend_entries:
            legend_entries[method] = sc

    x_concat = np.concatenate(all_x)
    y_concat = np.concatenate(all_y)
    lim = max(x_concat.max(), y_concat.max()) * 1.05
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)


    ax.set_title(f'{cell_type}', fontsize=15)
    ax.set_xlabel("Visium", fontsize=15)
    ax.set_ylabel("2x2 split Visium", fontsize=15)
    ax.tick_params(axis='x', labelsize=15)
    ax.tick_params(axis='y', labelsize=15)
    ax.plot([0, lim], [0, lim], ls='--', color='gray')

fig.legend(
    legend_entries.values(),
    legend_entries.keys(),
    loc='lower center',
    ncol=len(methods),
    fontsize=20,
    markerscale=2 
)


plt.suptitle("Comparison celltype proportions:\nVisium vs. 2x2 split Visium", fontsize=20)
plt.tight_layout(rect=[0, 0.05, 1, 0.99])
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

methods = ["nnls", "rctd", "cell2loc"]

# Alleen part 0 gebruiken
part0_mask = sdata.table.obs["part"] == 0

# Alleen de gewenste celtypes
cell_types = ["Cholangiocytes", "Hepatocytes", "Tcells"]

n_cols = 3
n_rows = 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5))
axes = axes.flatten()

# Verzamel legend handles en labels
legend_entries = {}

for idx, cell_type in enumerate(cell_types):
    ax = axes[idx]

    for method in methods:
        key_visium = f"visium_{method}"
        key_split = f"visium_{method}_2x2"

        df_visium = sdata.table.obsm[key_visium].iloc[part0_mask.values, :-1]
        df_split = sdata.table.obsm[key_split].iloc[part0_mask.values, :-1]

        x = df_visium[cell_type].astype(float)
        y = df_split[cell_type].astype(float)

        sc = ax.scatter(x, y, alpha=0.6, label=method, s=20)

        if method not in legend_entries:
            legend_entries[method] = sc

    # Titel en labels groter
    ax.set_title(f'{cell_type}', fontsize=20)
    ax.set_xlabel("Visium", fontsize=18)
    ax.set_ylabel("2x2 split Visium", fontsize=18)

    ax.tick_params(axis='both', labelsize=12)
    ax.plot([0, 1], [0, 1], ls='--', color='gray')

    # Beperk assen voor cholangiocytes
    if cell_type == "Cholangiocytes":
        ax.set_xlim(0, 0.4)
        ax.set_ylim(0, 0.4)

    # Info-kader met mean/std
    stats_text = "\n".join(
        f"{method}: mean={np.mean(sdata.table.obsm[f'ratio_cell_types_{method}'][cell_type]):.2f}, std={np.std(sdata.table.obsm[f'ratio_cell_types_{method}'][cell_type]):.2f}"
        for method in methods
    )

    ax.text(
        0.05, 0.95,
        stats_text,
        transform=ax.transAxes,
        fontsize=14,
        verticalalignment='top',
        bbox=dict(facecolor='white', alpha=0.2, edgecolor='black')
    )

# Verberg ongebruikte subplot-assen (indien nodig)
for k in range(len(cell_types), len(axes)):
    axes[k].axis('off')

# Legenda onderaan, groter font
#fig.legend(legend_entries.values(), legend_entries.keys(), loc='lower center', ncol=len(methods), fontsize=20)
fig.legend(
    legend_entries.values(),
    legend_entries.keys(),
    loc='lower center',
    bbox_to_anchor=(0.5, -0.1),  # 0.5 = horizontaal gecentreerd, -0.05 = lager onder de plot
    ncol=len(methods),
    fontsize=20
)

plt.suptitle("Comparison celltype proportions: Visium vs. 2x2 split Visium", fontsize=20)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()



In [ ]:
df_visium = sdata.table.obsm[key_visium].iloc[:,:-1]
df_visium